# Scout Shopping Assistant Project 1

One Gemini-powered LangGraph agent with a sample CSV tool, checkpointed approval, and approved cross-thread memory. This notebook imports the same `shopping_assistant.py` implementation used by the webapp. The implementation has lab-style `# %%` sections and small node functions.

The executed cells below use **Sample engine (no LLM)** and scripted mock-human decisions to make this walkthrough repeatable. Use the explicit live option below or the webapp to call Gemini. All catalog records and purchases are fictional or simulated.

Keep this notebook beside the Python files and CSV. Install once with `%pip install -r requirements.txt` in a separate cell, then restart the kernel and run all cells in order. Python 3.12 was used for verification.


## Imports and the same application

StateGraph controls execution. InMemorySaver holds per-thread checkpoints; InMemoryStore holds approved preferences for reuse by a new thread.

In [1]:
from shopping_assistant import ShoppingService, records
from run_demo import demonstrate

service = ShoppingService()
USER_ID = 'notebook-student'
print('Engine: Sample engine (no LLM). Data: fictional headphones.csv.')
print('Catalog offers:', len(__import__('shopping_assistant').read_catalog()))


Engine: Sample engine (no LLM). Data: fictional headphones.csv.
Catalog offers: 110


## Actual LangGraph schema

The frontend Architecture tab renders these same nodes and edges. The Mermaid definition is generated from the compiled graph, rather than a separately invented workflow.

In [2]:
print(service.architecture()['mermaid'])


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	intake(intake)
	understand(understand)
	retrieve_memory(retrieve_memory)
	clarify(clarify)
	plan_search(plan_search)
	search_catalog(search_catalog)
	compare_offers(compare_offers)
	recommend(recommend)
	approve_purchase(approve_purchase)
	confirm_purchase(confirm_purchase)
	cancel_purchase(cancel_purchase)
	no_matches(no_matches)
	unsupported(unsupported)
	insufficient(insufficient)
	form_memory(form_memory)
	approve_memory(approve_memory)
	save_memory(save_memory)
	finish_memory(finish_memory)
	__end__([<p>__end__</p>]):::last
	__start__ --> intake;
	approve_memory -. &nbsp;False&nbsp; .-> finish_memory;
	approve_memory -. &nbsp;True&nbsp; .-> save_memory;
	approve_purchase -. &nbsp;False&nbsp; .-> cancel_purchase;
	approve_purchase -. &nbsp;True&nbsp; .-> confirm_purchase;
	clarify --> understand;
	compare_offers -.-> no_matches;
	compare_offers -.-> recommend;
	form_memory -. &nbsp;True

## Required scenarios

The harness supplies explicit mock-human approval or rejection through actual `interrupt()` and `Command(resume=...)` calls. A normal request is followed by two approved preferences, a new shopping thread, rejection, correction, a temporary exception, and a no-match request.

In [3]:
results, checks = demonstrate(service, engine='sample', user_id=USER_ID)
assert all('FAIL' not in line for line in checks)


PASS normal_paused: expected=WAITING_APPROVAL; actual=WAITING_APPROVAL; thread=b0b43e33; engine=Sample engine (no LLM)
PASS normal_approved: expected=SIMULATED_CONFIRMED; actual=SIMULATED_CONFIRMED; thread=b0b43e33; engine=Sample engine (no LLM)
PASS memory_fit_paused: expected=MEMORY_PROPOSED; actual=MEMORY_PROPOSED; thread=c7e8a6e7; engine=Sample engine (no LLM)
PASS memory_fit_saved: expected=MEMORY_SAVED; actual=MEMORY_SAVED; thread=c7e8a6e7; engine=Sample engine (no LLM)
PASS memory_condition_paused: expected=MEMORY_PROPOSED; actual=MEMORY_PROPOSED; thread=7f1433dc; engine=Sample engine (no LLM)
PASS memory_condition_saved: expected=MEMORY_SAVED; actual=MEMORY_SAVED; thread=7f1433dc; engine=Sample engine (no LLM)
PASS later_using_two_memories: expected=WAITING_APPROVAL; actual=WAITING_APPROVAL; thread=747e18ed; engine=Sample engine (no LLM)
PASS purchase_rejected: expected=CANCELLED; actual=CANCELLED; thread=747e18ed; engine=Sample engine (no LLM)
PASS correction_paused: expected=

## Inspect the paused state and comparison

A separate state object and thread ID exist for each request. In Gemini mode, compare_offers calls the model with the request, applied preferences, search results, and validated shortlist to produce comparison bullets. Recommend receives that comparison and the request/preferences, selects one eligible offer, and writes a paragraph explaining why it wins. The sample engine below uses local rules with the same output structure. At purchase approval, no confirmation has occurred and the next node is `approve_purchase`.

In [4]:
paused = results['later_using_two_memories']
print('Thread:', paused['thread_id'])
print('Waiting node:', paused['next'])
print('Confirmed before approval:', paused['confirmed'])
print('Retrieved keys:', [m['key'] for m in paused['selected_memories']])
print('Effective constraints:', paused['effective_constraints'])
print('Rejected outcome:', results['purchase_rejected']['status'])
for offer in paused['comparison']['analysis']['offers']:
    print('Compared offer:', offer['offer_id'])
    for bullet in offer['bullets']:
        print(' -', bullet)
print('Why this product:', paused['recommendation']['explanation'])


Thread: 747e18edb0ad4b4a9f4c185b2bd59360
Waiting node: ['approve_purchase']
Confirmed before approval: False
Retrieved keys: ['condition', 'form_factor']
Effective constraints: {'category': 'headphones', 'budget': 150.0, 'form_factor': 'over-ear', 'condition': 'new', 'noise_cancelling': True, 'min_battery_hours': None, 'exact_model': None, 'priority': 'price', 'use_case': ''}
Rejected outcome: CANCELLED
Compared offer: O072
 - Lowest known delivered cost in this shortlist.
 - 42 hours of listed battery life; over-ear fit and new condition.
 - Meets the active requirements, with stock and shipping confirmed in the sample CSV.
Compared offer: O098
 - Costs more than the lowest-price option in this shortlist.
 - 50 hours of listed battery life; over-ear fit and new condition.
 - Meets the active requirements, with stock and shipping confirmed in the sample CSV.
Compared offer: O038
 - Costs more than the lowest-price option in this shortlist.
 - 60 hours of listed battery life; over-ear f

## Memory use and correction

Both records are formed in earlier threads. A correction supersedes the condition preference after approval while preserving the original provenance. A purchase-specific exception is not a new general preference.

In [5]:
print('Without memory:', results['normal_paused']['recommendation']['offer_id'])
print('With two memories:', results['later_using_two_memories']['recommendation']['offer_id'])
print('After correction:', results['later_after_correction']['effective_constraints']['condition'])
for memory in records(service.store, USER_ID, history=True):
    print(memory['key'], memory['value'], memory['status'], 'source:', memory['source_thread'])
print('Temporary input:', results['temporary_not_saved']['status'])


Without memory: O050
With two memories: O072
After correction: any
form_factor over-ear active source: c7e8a6e74014472bbc38dca3790379e1
condition new superseded source: 7f1433dc62aa48aba292cac2f61ab20f
condition any active source: 4c9512f99dea431a999ca731f4448ec1
Temporary input: NOT_STORED


## Regression verification

These checks use sample rules or explicit invalid-output stubs. They exercise graph control, money calculations, memory boundaries, and HTTP safeguards, rather than estimating Gemini accuracy.

In [6]:
import unittest
import sys
import project_tests
suite = unittest.defaultTestLoader.loadTestsFromModule(project_tests)
test_result = unittest.TextTestRunner(stream=sys.stdout, verbosity=1).run(suite)
assert test_result.wasSuccessful()


...............................
----------------------------------------------------------------------
Ran 31 tests in 0.441s

OK


## Explicit live Gemini option

Set `RUN_LIVE = True` to use a working Gemini model. The hidden prompt reads a key into process memory. The live result pauses at real user approval and does not automatically confirm a purchase. Do not save API keys in notebook source or outputs.


In [7]:
RUN_LIVE = False
if RUN_LIVE:
    import getpass
    live_service = ShoppingService(getpass.getpass('Google API key (hidden): '))
    live_result = live_service.start('live-student', 'Find new over-ear headphones under $150 with noise cancellation.')
    print('Status:', live_result['status'])
    print('Pending:', live_result.get('pending'))
else:
    print('Live cell disabled. Enable explicitly or use the webapp Gemini engine.')


Live cell disabled. Enable explicitly or use the webapp Gemini engine.


## Scope and limitations

Memory and unfinished checkpoints clear on process or kernel restart. Simulated user IDs demonstrate scoped access, not production authentication. Product lookup is CSV-only; costs exclude tax and live verification. Gemini errors remain errors and never silently select sample rules. See `README.md` and the two-page report for setup, test evidence, limitations, and the contribution statement.
